In [0]:
# 03_ml_comparison

import json
import time
import uuid
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


CATALOG = "workspace"
SCHEMA = "stroke_prediction"

TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.stroke_train"
MODEL_COMPARISON_TABLE = f"{CATALOG}.{SCHEMA}.ml_model_comparison"
PARAMETER_RESULTS_TABLE = f"{CATALOG}.{SCHEMA}.ml_parameter_results"
SELECTED_MODEL_TABLE = f"{CATALOG}.{SCHEMA}.ml_selected_model"

TARGET_COLUMN = "stroke"

RANDOM_SEED = 42
CV_FOLDS = 5
N_JOBS = -1
SELECTION_METRIC = "average_precision"

CONTINUOUS_COLUMNS = [
    "age",
    "bmi",
    "avg_glucose_level",
]

BINARY_COLUMNS = [
    "hypertension",
    "heart_disease",
]

CATEGORICAL_COLUMNS = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status",
]

FEATURE_COLUMNS = (
    CONTINUOUS_COLUMNS
    + BINARY_COLUMNS
    + CATEGORICAL_COLUMNS
)


def new_rng():
    return np.random.RandomState(RANDOM_SEED)


def create_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


def create_preprocessor():
    continuous_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    binary_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", create_one_hot_encoder()),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "continuous",
                continuous_pipeline,
                CONTINUOUS_COLUMNS,
            ),
            (
                "binary",
                binary_pipeline,
                BINARY_COLUMNS,
            ),
            (
                "categorical",
                categorical_pipeline,
                CATEGORICAL_COLUMNS,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def parameters_to_json(parameters):
    def convert_value(value):
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, np.ndarray):
            return value.tolist()
        return str(value)

    return json.dumps(
        parameters,
        sort_keys=True,
        default=convert_value,
    )


def safe_float(value):
    if pd.isna(value):
        return float("nan")
    return float(value)


def write_table(dataframe, table_name):
    if dataframe.empty:
        raise ValueError(
            f"Cannot write an empty DataFrame to {table_name}."
        )

    (
        spark.createDataFrame(dataframe)
        .write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )


train_data = spark.table(TRAIN_TABLE).toPandas()

required_columns = set(
    FEATURE_COLUMNS + [TARGET_COLUMN]
)

missing_columns = sorted(
    required_columns.difference(train_data.columns)
)

if missing_columns:
    raise ValueError(
        "The training table is missing required columns: "
        f"{missing_columns}"
    )

if train_data[TARGET_COLUMN].isna().any():
    raise ValueError(
        "Missing target values were found in the training table."
    )

X_train = train_data[FEATURE_COLUMNS].copy()

y_train = pd.to_numeric(
    train_data[TARGET_COLUMN],
    errors="raise",
).astype(int)

if not set(y_train.unique()).issubset({0, 1}):
    raise ValueError(
        "The stroke target must contain only 0 and 1. "
        f"Found: {sorted(y_train.unique())}"
    )

for column in CONTINUOUS_COLUMNS + BINARY_COLUMNS:
    X_train[column] = pd.to_numeric(
        X_train[column],
        errors="coerce",
    )

for column in CATEGORICAL_COLUMNS:
    X_train[column] = (
        X_train[column]
        .where(X_train[column].notna(), np.nan)
        .astype(object)
    )

print(f"Training table: {TRAIN_TABLE}")
print(f"Training rows: {len(train_data):,}")
print(f"Stroke cases: {int(y_train.sum()):,}")
print(f"Stroke prevalence: {y_train.mean():.2%}")
print(f"scikit-learn version: {sklearn.__version__}")


SCORING = {
    "average_precision": "average_precision",
    "roc_auc": "roc_auc",
    "recall": make_scorer(
        recall_score,
        pos_label=1,
        zero_division=0,
    ),
    "precision": make_scorer(
        precision_score,
        pos_label=1,
        zero_division=0,
    ),
    "f1": make_scorer(
        f1_score,
        pos_label=1,
        zero_division=0,
    ),
    "specificity": make_scorer(
        recall_score,
        pos_label=0,
        zero_division=0,
    ),
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
}

cross_validation = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED,
)


MODEL_SPECS = {
    "dummy": {
        "name": "Dummy baseline",
        "family": "Baseline",
        "description": (
            "Baseline using the class distribution in the training data."
        ),
        "estimator": DummyClassifier(
            strategy="prior",
        ),
        "parameters": {},
    },

    "logistic_regression": {
        "name": "Logistic regression",
        "family": "Linear model",
        "description": (
            "Regularised linear classification model."
        ),
        "estimator": LogisticRegression(
            solver="liblinear",
            max_iter=2_000,
            random_state=new_rng(),
        ),
        "parameters": {
            "model__C": [
                0.1,
                1.0,
                10.0,
            ],
            "model__class_weight": [
                None,
                "balanced",
            ],
        },
    },

    "decision_tree": {
        "name": "Decision tree",
        "family": "Tree",
        "description": (
            "Single classification tree."
        ),
        "estimator": DecisionTreeClassifier(
            random_state=new_rng(),
        ),
        "parameters": {
            "model__criterion": [
                "gini",
                "entropy",
            ],
            "model__max_depth": [
                3,
                6,
                None,
            ],
            "model__min_samples_leaf": [
                1,
                10,
            ],
            "model__class_weight": [
                None,
                "balanced",
            ],
        },
    },

    "random_forest": {
        "name": "Random forest",
        "family": "Bagged trees",
        "description": (
            "Ensemble of decision trees fitted to bootstrap samples."
        ),
        "estimator": RandomForestClassifier(
            random_state=new_rng(),
            n_jobs=1,
        ),
        "parameters": {
            "model__n_estimators": [
                200,
            ],
            "model__max_depth": [
                5,
                10,
                None,
            ],
            "model__min_samples_leaf": [
                1,
                5,
            ],
            "model__class_weight": [
                None,
                "balanced",
            ],
        },
    },

    "support_vector_machine": {
        "name": "Support vector machine",
        "family": "Kernel model",
        "description": (
            "Nonlinear support vector classifier using an RBF kernel."
        ),
        "estimator": SVC(
            kernel="rbf",
            gamma="scale",
            probability=False,
        ),
        "parameters": {
            "model__C": [
                0.1,
                1.0,
                10.0,
            ],
            "model__class_weight": [
                None,
                "balanced",
            ],
        },
    },

    "gradient_boosting": {
        "name": "Gradient boosting",
        "family": "Boosted trees",
        "description": (
            "Sequential ensemble of shallow decision trees."
        ),
        "estimator": GradientBoostingClassifier(
            random_state=new_rng(),
        ),
        "parameters": {
            "model__n_estimators": [
                100,
                200,
            ],
            "model__learning_rate": [
                0.05,
                0.1,
            ],
            "model__max_depth": [
                1,
                3,
            ],
        },
    },
}


run_id = uuid.uuid4().hex
run_timestamp_utc = datetime.now(
    timezone.utc
).isoformat()

model_results = []
parameter_results = []

for model_key, model_config in MODEL_SPECS.items():
    model_name = model_config["name"]

    print()
    print("=" * 72)
    print(f"Running: {model_name}")
    print("=" * 72)

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                create_preprocessor(),
            ),
            (
                "model",
                model_config["estimator"],
            ),
        ]
    )

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=model_config["parameters"],
        scoring=SCORING,
        refit=SELECTION_METRIC,
        cv=cross_validation,
        n_jobs=N_JOBS,
        pre_dispatch="2*n_jobs",
        return_train_score=True,
        error_score="raise",
        verbose=1,
    )

    start_time = time.perf_counter()

    grid_search.fit(
        X_train,
        y_train,
    )

    search_seconds = (
        time.perf_counter() - start_time
    )

    cv_results = pd.DataFrame(
        grid_search.cv_results_
    )

    best_result = cv_results.iloc[
        int(grid_search.best_index_)
    ]

    model_result = {
        "run_id": run_id,
        "run_timestamp_utc": run_timestamp_utc,
        "training_table": TRAIN_TABLE,
        "model_key": model_key,
        "model_name": model_name,
        "model_family": model_config["family"],
        "description": model_config["description"],
        "selection_metric": SELECTION_METRIC,
        "best_parameters": parameters_to_json(
            grid_search.best_params_
        ),
        "parameter_candidates": int(len(cv_results)),
        "cv_folds": int(CV_FOLDS),
        "search_seconds": float(search_seconds),
        "training_rows": int(len(y_train)),
        "stroke_cases": int(y_train.sum()),
        "stroke_prevalence": float(y_train.mean()),
        "sklearn_version": sklearn.__version__,
    }

    for metric_name in SCORING:
        model_result[
            f"cv_{metric_name}_mean"
        ] = safe_float(
            best_result[
                f"mean_test_{metric_name}"
            ]
        )

        model_result[
            f"cv_{metric_name}_std"
        ] = safe_float(
            best_result[
                f"std_test_{metric_name}"
            ]
        )

        model_result[
            f"train_{metric_name}_mean"
        ] = safe_float(
            best_result[
                f"mean_train_{metric_name}"
            ]
        )

    model_results.append(model_result)

    for candidate_index, row in cv_results.iterrows():
        parameter_result = {
            "run_id": run_id,
            "run_timestamp_utc": run_timestamp_utc,
            "model_key": model_key,
            "model_name": model_name,
            "model_family": model_config["family"],
            "candidate_number": int(candidate_index + 1),
            "is_best_for_model": bool(
                candidate_index == grid_search.best_index_
            ),
            "parameters": parameters_to_json(
                row["params"]
            ),
            "average_precision_rank": int(
                row["rank_test_average_precision"]
            ),
            "mean_fit_time_seconds": safe_float(
                row["mean_fit_time"]
            ),
            "std_fit_time_seconds": safe_float(
                row["std_fit_time"]
            ),
            "mean_score_time_seconds": safe_float(
                row["mean_score_time"]
            ),
            "std_score_time_seconds": safe_float(
                row["std_score_time"]
            ),
            "cv_folds": int(CV_FOLDS),
            "sklearn_version": sklearn.__version__,
        }

        for metric_name in SCORING:
            parameter_result[
                f"cv_{metric_name}_mean"
            ] = safe_float(
                row[
                    f"mean_test_{metric_name}"
                ]
            )

            parameter_result[
                f"cv_{metric_name}_std"
            ] = safe_float(
                row[
                    f"std_test_{metric_name}"
                ]
            )

            parameter_result[
                f"train_{metric_name}_mean"
            ] = safe_float(
                row[
                    f"mean_train_{metric_name}"
                ]
            )

            parameter_result[
                f"train_{metric_name}_std"
            ] = safe_float(
                row[
                    f"std_train_{metric_name}"
                ]
            )

        parameter_results.append(
            parameter_result
        )

    print(
        "Best parameters:",
        grid_search.best_params_,
    )
    print(
        "Best CV average precision:",
        round(grid_search.best_score_, 4),
    )
    print(
        "Search time:",
        round(search_seconds, 1),
        "seconds",
    )


model_comparison = pd.DataFrame(
    model_results
)

parameter_comparison = pd.DataFrame(
    parameter_results
)

model_comparison = (
    model_comparison
    .sort_values(
        "cv_average_precision_mean",
        ascending=False,
    )
    .reset_index(drop=True)
)

model_comparison.insert(
    0,
    "overall_rank",
    range(
        1,
        len(model_comparison) + 1,
    ),
)

selected_model = (
    model_comparison
    .head(1)
    .copy()
)

selected_model.insert(
    1,
    "selected",
    True,
)

write_table(
    model_comparison,
    MODEL_COMPARISON_TABLE,
)

write_table(
    parameter_comparison,
    PARAMETER_RESULTS_TABLE,
)

write_table(
    selected_model,
    SELECTED_MODEL_TABLE,
)

print()
print(f"Saved: {MODEL_COMPARISON_TABLE}")
print(f"Saved: {PARAMETER_RESULTS_TABLE}")
print(f"Saved: {SELECTED_MODEL_TABLE}")


display(
    model_comparison[
        [
            "overall_rank",
            "model_name",
            "model_family",
            "cv_average_precision_mean",
            "cv_average_precision_std",
            "cv_roc_auc_mean",
            "cv_recall_mean",
            "cv_precision_mean",
            "cv_f1_mean",
            "cv_balanced_accuracy_mean",
            "cv_specificity_mean",
            "cv_accuracy_mean",
            "train_average_precision_mean",
            "best_parameters",
            "search_seconds",
        ]
    ]
)

display(
    parameter_comparison
    .sort_values(
        [
            "model_name",
            "average_precision_rank",
        ]
    )
)
